# Laboratory notebook - reproducing every figure of
## *Integrative transcriptomic dynamics in HPV-driven neoplasia and cervical cancer progression*

This notebook documents, step by step and with comments, how each figure and table of the manuscript is generated in **R**.
Every code cell is the *exact* script that was run to produce the reported results (`code/cells/*.R`), so the notebook and the
manuscript cannot drift apart.

| Figure | Content | Dataset | Cell |
|---|---|---|---|
| Fig. 1 | Volcano plot, Cancer vs Normal (limma) | GSE63514 | 3 |
| Fig. 2 | GO Biological Process over-representation | GSE63514 | 4 |
| Fig. 3 | 18-gene panel along Normal -> CIN1 -> CIN2 -> CIN3 -> Cancer | GSE63514 | 5 |
| Fig. 4 | Volcano plot, Tumour vs Normal (limma-trend) | TCGA-CESC | 8 |
| Fig. 5 | Heatmap of the 18-gene panel annotated with histology, FIGO stage and HPV genotype | TCGA-CESC | 9 |
| Fig. 6 | Overall survival: Kaplan-Meier for CD274 and Cox forest plot (22 transcripts) | TCGA-CESC | 10 |
| Fig. 7 | DNA methylation (HM450): volcano and CpG-island context | TCGA-CESC | 11 |
| Table S15 | Sensitivity of the panel to the probe-collapsing rule (highest mean vs highest variance) | GSE63514 | 12 |

**How to run.** Open R (>= 4.4) with the working directory set to the project root (the folder that contains `code/`, `data/`, `results/`),
or use the R kernel (`IRkernel`) in Jupyter. Execute the cells in order. The first run downloads GSE63514 from GEO (~50 MB) and
TCGA-CESC from the GDC (~1.3 GB, 309 files; Cell 11 additionally downloads 312 HM450 beta-value files, ~3 GB, and the GPL13534 annotation); both are cached under `data/` so later runs are offline-safe.

**Random seed.** `set.seed(2026)` is set in Cell 1; no analysis below is stochastic, but the seed is fixed for completeness.

**Not executed here.** Output cells are intentionally empty: the notebook is a *recipe*. The numbers quoted in the paper were
captured by the same scripts into `results/tables/numbers_logged_all.csv` (see Cell 13).

**File names vs manuscript numbering.** The scripts write files named `Fig2_*`, `Fig3_*`, `Fig4_*`, `Fig6_*` and `Fig5_*`. In the manuscript these are
**Fig. 1** (`Fig2_volcano_GSE63514`), **Fig. 2** (`Fig3_GO_BP_GSE63514`), **Fig. 3** (`Fig4_panel_GSE63514`), **Fig. 4** (`Fig6_volcano_TCGA_CESC`) and **Fig. 5** (`Fig5_heatmap_TCGA_CESC`).
Cell 8 (TCGA volcano) is placed after Cell 7 in this notebook; it only needs `expr_t` and `tcga_group` from Cell 6 (and `deg` from Cell 3 for the concordance numbers).


## Cell 0 - Package installation (run once)
Bioconductor 3.23 / R 4.6.1 were used for the reported results. Packages: GEOquery (GEO access), limma (linear models with empirical-Bayes
moderation; Ritchie et al., *Nucleic Acids Res* 2015, PMID 25605792), clusterProfiler (enrichment; Wu et al., *Innovation* 2021, PMID 34557778),
TCGAbiolinks (GDC access; Colaprico et al., *Nucleic Acids Res* 2016, PMID 26704973).

In [ ]:
# Run once. Skips packages that are already installed.
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
bioc <- c("GEOquery", "Biobase", "limma", "clusterProfiler", "org.Hs.eg.db", "TCGAbiolinks", "SummarizedExperiment")
cran <- c("dplyr", "tidyr", "tibble", "ggplot2", "ggrepel", "pheatmap", "RColorBrewer", "matrixStats")
need <- c(bioc[!bioc %in% rownames(installed.packages())], cran[!cran %in% rownames(installed.packages())])
if (length(need)) BiocManager::install(need, ask = FALSE, update = FALSE)


## Cell 1 - Environment, thresholds and helpers
Defines every analysis parameter in one place (FDR < 0.05, |log2FC| > 1) and the **exclusion rule**: mitochondrially-encoded (`MT-`) and cytosolic
ribosomal-protein transcripts are removed *before* differential expression and clustering.

*Why, and what it costs.* These transcripts dominate library composition and are sensitive to tissue handling, so they were removed as a
reviewer-requested quality step. Note, however, that den Boon et al. (*PNAS* 2015, PMID 26056290), who generated GSE63514, report **lower expression of
mitochondrial electron-transport and ribosomal-protein genes at the CIN3-to-cancer transition as a biological finding**. The removal is therefore a
deliberate analytical choice, not a statement that these genes are uninformative; a sensitivity analysis with them retained is in Cell 3.
The regular expression was checked to drop ribosomal-protein genes (e.g. *RPL10A*, *RPS4X*) while keeping the kinases *RPS6KA1/RPS6KB1*,
`RPL/RPS` pseudogenes and mitochondrial ribosomal genes (*MRPL/MRPS*).

In [ ]:
# ==============================================================================
# Cell 01 - Environment, global parameters and helper functions
# Working directory must be the project root (the folder that contains code/).
# ==============================================================================
suppressPackageStartupMessages({
  library(GEOquery); library(Biobase); library(limma)
  library(dplyr); library(tidyr); library(tibble)
  library(ggplot2); library(ggrepel); library(pheatmap)
  library(RColorBrewer); library(matrixStats)
  library(clusterProfiler); library(org.Hs.eg.db)
})
set.seed(2026)                                   # fixed seed used in every script
options(timeout = 1200)                          # large GEO/GDC downloads
for (d in c("data", "results/figures", "results/tables"))
  dir.create(d, recursive = TRUE, showWarnings = FALSE)

# ---- Thresholds (identical to the Methods section of the manuscript) ----------
FDR_CUT <- 0.05          # Benjamini-Hochberg FDR
LFC_CUT <- 1             # |log2 fold change| threshold for the volcano / GO input

# ---- Histological groups of GSE63514 -----------------------------------------
GROUP_LEVELS <- c("Normal", "CIN1", "CIN2", "CIN3", "Cancer")
GROUP_COLS   <- c(Normal = "#4C78A8", CIN1 = "#72B7B2", CIN2 = "#F2CF5B",
                  CIN3 = "#F58518", Cancer = "#E45756")

# ---- Mitochondrially-encoded (MT-) and cytosolic ribosomal-protein transcripts
# are removed BEFORE any differential-expression / clustering step because they
# are dominated by pre-analytical (bench/tissue-quality) variation.
# MRPL*/MRPS* (nuclear-encoded mitochondrial ribosomal proteins) are retained.
EXCLUDE_REGEX <- "^MT-|^RP[SL][0-9]+[AXY]?[0-9]?L?[0-9]?$|^RPLP[0-2]$|^RPSA$"
# Pattern checked to EXCLUDE ribosomal-protein genes (RPL10A, RPS4X, RPL36AL, RPL22L1...) but KEEP kinases
# (RPS6KA1, RPS6KB1), RPL/RPS pseudogenes (RPL21P28) and mitochondrial ribosomal genes (MRPL*, MRPS*).

# ---- Registry of every number quoted in the manuscript text -------------------
NUM_LOG <- list()
log_num <- function(key, value) { NUM_LOG[[key]] <<- value; invisible(value) }

# ---- Figure writer: 300 dpi PNG + vector PDF + LZW-compressed TIFF ------------
save_fig <- function(p, name, w = 7, h = 5) {
  base <- file.path("results/figures", name)
  ggsave(paste0(base, ".png"),  p, width = w, height = h, dpi = 300, type = "cairo")
  ggsave(paste0(base, ".pdf"),  p, width = w, height = h, device = "pdf")
  ggsave(paste0(base, ".tiff"), p, width = w, height = h, dpi = 300, compression = "lzw")
  invisible(p)
}
fmt_p <- function(p) ifelse(p < 1e-3, formatC(p, format = "e", digits = 1), formatC(p, format = "f", digits = 3))
stars <- function(p) ifelse(p < 0.001, "***", ifelse(p < 0.01, "**", ifelse(p < 0.05, "*", "ns")))
theme_paper <- theme_classic(base_size = 11) +
  theme(plot.title = element_text(face = "bold"), strip.background = element_blank(),
        strip.text = element_text(face = "bold"))
cat("Setup OK | R", as.character(getRversion()), "| Bioconductor", as.character(BiocManager::version()), "\n")

## Cell 2 - GSE63514: load, define groups, collapse probes to genes
GSE63514 profiles 128 cervical samples on Affymetrix U133 Plus 2.0 (GPL570): 24 normal, 14 CIN1, 22 CIN2, 40 CIN3, 28 cancer.

1. The matrix is already log2-scaled (maximum ~17), so no further transform is applied; the script only transforms if the maximum exceeds 100.
2. Histological group is parsed from the `tissue type` characteristic.
3. Probes without a symbol or mapped to several genes (`///`) are dropped, because their signal cannot be assigned to one gene.
4. When several probes map to one gene, the probe with the **highest mean expression** is kept (a standard, deterministic collapsing rule).
5. MT-/ribosomal genes are removed and the excluded list is saved (`TableS_excluded_genes_GSE63514.csv`).

Dissection method (laser capture vs whole section) is stored as a technical covariate for the sensitivity analysis in Cell 3: all 8 whole-section samples are cancers.

In [ ]:
# ==============================================================================
# Cell 02 - GSE63514: download, group parsing, probe-to-gene collapsing
# Affymetrix Human Genome U133 Plus 2.0 (GPL570); den Boon et al., PNAS 2015
# ==============================================================================
# Cached ExpressionSet is used when present (offline-safe); otherwise it is downloaded from GEO and cached.
if (file.exists("data/gse63514_eset.rds")) {
  eset <- readRDS("data/gse63514_eset.rds")
} else {
  eset <- getGEO("GSE63514", GSEMatrix = TRUE, AnnotGPL = TRUE, destdir = "data")[[1]]
  saveRDS(eset, "data/gse63514_eset.rds")
}
expr_probe <- exprs(eset)
log_num("gse63514_n_probes_raw", nrow(expr_probe))
log_num("gse63514_max_intensity", signif(max(expr_probe, na.rm = TRUE), 5))

# Scale check (Methods 2.3): log2(x + 1) only if the matrix is not already on log scale
if (max(expr_probe, na.rm = TRUE) > 100) expr_probe <- log2(expr_probe + 1)

# ---- Histological group from the sample characteristics ----------------------
pd <- pData(eset)
tt <- pd[["tissue type:ch1"]]
group <- factor(dplyr::case_when(
  grepl("normal",         tt) ~ "Normal",
  grepl("low grade",      tt) ~ "CIN1",
  grepl("moderate grade", tt) ~ "CIN2",
  grepl("high grade",     tt) ~ "CIN3",
  grepl("cancer",         tt) ~ "Cancer"), levels = GROUP_LEVELS)
stopifnot(!anyNA(group))
print(table(group))
for (g in GROUP_LEVELS) log_num(paste0("gse63514_n_", g), sum(group == g))

# Dissection method (technical covariate used only in a sensitivity analysis)
dissection <- factor(ifelse(grepl("whole", pd[["dissection:ch1"]]), "whole_section", "laser_capture"))
print(table(group, dissection))

# ---- Probe -> HGNC symbol; multi-gene ("///") and unannotated probes dropped ---
sym  <- trimws(fData(eset)[["Gene symbol"]])
keep <- !is.na(sym) & sym != "" & !grepl("///", sym, fixed = TRUE)
log_num("gse63514_n_probes_dropped_no_or_multi_symbol", sum(!keep))
probe_tab <- tibble(probe = rownames(expr_probe)[keep], symbol = sym[keep],
                    mean_expr = rowMeans(expr_probe[keep, ]))
# One probe per gene: the probe with the highest mean expression across samples
best <- probe_tab %>% group_by(symbol) %>% slice_max(mean_expr, n = 1, with_ties = FALSE) %>% ungroup()
expr_all <- expr_probe[best$probe, ]; rownames(expr_all) <- best$symbol
log_num("gse63514_n_genes_collapsed", nrow(expr_all))

# ---- Remove MT- and ribosomal-protein transcripts before DEG / clustering ------
excl <- grepl(EXCLUDE_REGEX, rownames(expr_all))
write.csv(data.frame(gene = rownames(expr_all)[excl]),
          "results/tables/TableS_excluded_genes_GSE63514.csv", row.names = FALSE)
log_num("gse63514_n_genes_excluded_MT_ribosomal", sum(excl))
log_num("gse63514_n_MT_genes_present", sum(grepl("^MT-", rownames(expr_all))))
expr <- expr_all[!excl, ]
log_num("gse63514_n_genes_analysed", nrow(expr))
cat(sprintf("GSE63514: %d genes after collapsing; %d MT-/ribosomal removed; %d analysed\n",
            nrow(expr_all), sum(excl), nrow(expr)))

## Cell 3 - Differential expression (limma) and **Fig. 1** (volcano)
`lmFit` fits one mean per histological group; `contrasts.fit` + `eBayes` give moderated t-statistics, and p-values are adjusted by Benjamini-Hochberg.
Genes are called up/down in cancer at FDR < 0.05 and |log2FC| > 1. The ten strongest up- and down-regulated genes are labelled (`Fig2_labelled_genes.csv`).

Two sensitivity analyses are logged: (a) keeping MT-/ribosomal genes, (b) adjusting for dissection method. Both feed the Supplement.

In [ ]:
# ==============================================================================
# Cell 03 - Differential expression (limma) and volcano plot (Fig. 2)
# ==============================================================================
run_limma <- function(mat, group, covariate = NULL) {
  design <- if (is.null(covariate)) model.matrix(~ 0 + group) else model.matrix(~ 0 + group + covariate)
  colnames(design) <- make.names(sub("^group", "", colnames(design)))
  fit <- lmFit(mat, design)
  cm  <- makeContrasts(Cancer_vs_Normal = Cancer - Normal, CIN1_vs_Normal = CIN1 - Normal,
                       CIN2_vs_Normal = CIN2 - Normal, CIN3_vs_Normal = CIN3 - Normal,
                       Cancer_vs_CIN3 = Cancer - CIN3, levels = design)
  eBayes(contrasts.fit(fit, cm))               # empirical-Bayes moderation
}
classify <- function(tab) dplyr::mutate(tab, class = dplyr::case_when(
  adj.P.Val < FDR_CUT & logFC >  LFC_CUT ~ "Up in cancer",
  adj.P.Val < FDR_CUT & logFC < -LFC_CUT ~ "Down in cancer", TRUE ~ "Not significant"))
get_tab <- function(fit, coef) topTable(fit, coef = coef, number = Inf, adjust.method = "BH", sort.by = "none") %>%
  tibble::rownames_to_column("gene")

fit  <- run_limma(expr, group)
deg  <- classify(get_tab(fit, "Cancer_vs_Normal"))
write.csv(deg, "results/tables/TableS2_GSE63514_limma_Cancer_vs_Normal.csv", row.names = FALSE)
for (cf in c("CIN1_vs_Normal", "CIN2_vs_Normal", "CIN3_vs_Normal", "Cancer_vs_CIN3"))
  write.csv(classify(get_tab(fit, cf)), sprintf("results/tables/TableS2_GSE63514_limma_%s.csv", cf), row.names = FALSE)

n_up <- sum(deg$class == "Up in cancer"); n_dn <- sum(deg$class == "Down in cancer")
log_num("deg_n_tested", nrow(deg)); log_num("deg_n_up", n_up); log_num("deg_n_down", n_dn)
log_num("deg_logFC_min", round(min(deg$logFC), 2)); log_num("deg_logFC_max", round(max(deg$logFC), 2))
log_num("deg_max_neglog10FDR", round(max(-log10(deg$adj.P.Val)), 2))
cat(sprintf("Cancer vs Normal: %d up, %d down (FDR<%.2f, |log2FC|>%d)\n", n_up, n_dn, FDR_CUT, LFC_CUT))

# ---- Sensitivity analyses (reported in the Supplement) ------------------------
# (a) with MT-/ribosomal transcripts retained
deg_with <- classify(get_tab(run_limma(expr_all, group), "Cancer_vs_Normal"))
sig_main <- deg$gene[deg$class != "Not significant"]; sig_with <- deg_with$gene[deg_with$class != "Not significant"]
log_num("sens_with_MTribo_n_deg", length(sig_with)); log_num("sens_with_MTribo_overlap", length(intersect(sig_main, sig_with)))
# (b) adjusting for dissection method (laser capture vs whole section)
fit_d <- run_limma(expr, group, covariate = dissection)
deg_d <- classify(get_tab(fit_d, "Cancer_vs_Normal"))
sig_d <- deg_d$gene[deg_d$class != "Not significant"]
log_num("sens_dissection_n_deg", length(sig_d)); log_num("sens_dissection_overlap", length(intersect(sig_main, sig_d)))
log_num("sens_dissection_logFC_cor", round(cor(deg$logFC, deg_d$logFC[match(deg$gene, deg_d$gene)]), 3))

# ---- Fig. 2: volcano with the 10 strongest up- and down-regulated genes labelled
lab <- bind_rows(deg %>% filter(class == "Up in cancer")   %>% slice_max(logFC, n = 10),
                 deg %>% filter(class == "Down in cancer") %>% slice_min(logFC, n = 10))
write.csv(lab, "results/tables/Fig2_labelled_genes.csv", row.names = FALSE)
cols <- c("Up in cancer" = "#D62728", "Down in cancer" = "#1F77B4", "Not significant" = "grey70")
p2 <- ggplot(deg, aes(logFC, -log10(adj.P.Val), colour = class)) +
  geom_point(size = 0.7, alpha = 0.6) +
  geom_vline(xintercept = c(-LFC_CUT, LFC_CUT), linetype = "dashed", colour = "grey40") +
  geom_hline(yintercept = -log10(FDR_CUT), linetype = "dashed", colour = "grey40") +
  geom_text_repel(data = lab, aes(label = gene), size = 2.8, colour = "black", fontface = "italic",
                  max.overlaps = Inf, box.padding = 0.35, segment.size = 0.25, min.segment.length = 0) +
  scale_colour_manual(values = cols, name = NULL) +
  labs(x = expression(log[2]~"fold change (Cancer / Normal)"), y = expression(-log[10]~"FDR")) +
  theme_paper + theme(legend.position = "bottom")
save_fig(p2, "Fig2_volcano_GSE63514", 7, 5.5)
print(p2)

## Cell 4 - GO Biological Process enrichment and **Fig. 2**
Over-representation (hypergeometric) of up-regulated genes against the **universe of genes measured on the array** (not the whole genome), which avoids the
classic bias of testing against unmeasured genes. Terms are required to contain 10-500 genes, p adjusted by BH (< 0.05), and redundant terms are merged with
`simplify()` (semantic similarity cut-off 0.7). The 15 most significant terms are plotted; the plotted gene ratio uses the number of input genes that have a GO-BP annotation.

In [ ]:
# ==============================================================================
# Cell 04 - GO Biological Process over-representation of up-regulated genes (Fig. 3)
# ==============================================================================
up_genes <- deg$gene[deg$class == "Up in cancer"]
map <- suppressMessages(bitr(rownames(expr), fromType = "SYMBOL", toType = "ENTREZID", OrgDb = org.Hs.eg.db))
universe <- unique(map$ENTREZID)                       # all genes measured on the array
up_entrez <- unique(map$ENTREZID[map$SYMBOL %in% up_genes])
ego <- enrichGO(gene = up_entrez, universe = universe, OrgDb = org.Hs.eg.db, keyType = "ENTREZID",
                ont = "BP", pAdjustMethod = "BH", pvalueCutoff = 0.05, qvalueCutoff = 0.05,
                minGSSize = 10, maxGSSize = 500, readable = TRUE)
ego_s <- simplify(ego, cutoff = 0.7, by = "p.adjust", select_fun = min)   # remove redundant terms
res <- as.data.frame(ego_s)
write.csv(as.data.frame(ego),  "results/tables/TableS3_GSE63514_GO_BP_all.csv",        row.names = FALSE)
write.csv(res,                 "results/tables/TableS3_GSE63514_GO_BP_simplified.csv", row.names = FALSE)
n_annot <- length(intersect(up_entrez, unique(AnnotationDbi::select(org.Hs.eg.db, up_entrez, "GOALL", "ENTREZID") %>%
                                              dplyr::filter(ONTOLOGYALL == "BP") %>% dplyr::pull(ENTREZID))))
log_num("go_n_input_genes", length(up_entrez)); log_num("go_n_input_with_BP", n_annot)
log_num("go_n_terms_raw", nrow(as.data.frame(ego))); log_num("go_n_terms_simplified", nrow(res))
top15 <- res %>% arrange(p.adjust) %>% slice_head(n = 15) %>%
  mutate(GeneRatioNum = Count / n_annot, Description = factor(Description, levels = rev(Description)))
write.csv(top15, "results/tables/Fig3_top15_GO_BP.csv", row.names = FALSE)
log_num("go_top15_max_padj", signif(max(top15$p.adjust), 3)); log_num("go_top1_term", as.character(top15$Description[1]))
log_num("go_top1_padj", signif(top15$p.adjust[1], 3))
p3 <- ggplot(top15, aes(GeneRatioNum, Description, size = Count, colour = p.adjust)) +
  geom_point() + scale_colour_viridis_c(option = "plasma", direction = -1, name = "Adjusted P") +
  scale_size_continuous(name = "Genes") +
  labs(x = "Gene ratio", y = "Biological processes") + theme_paper
save_fig(p3, "Fig3_GO_BP_GSE63514", 8, 5)
print(p3)

## Cell 5 - 18-gene panel and **Fig. 3**
Eighteen genes were chosen a priori to cover five axes: proliferation/DNA replication, HPV/cell-cycle control, oncogenic signalling, squamous differentiation and
hypoxia/angiogenesis/immune evasion. For each gene: one-way ANOVA across the five groups, Tukey HSD post hoc comparisons (reported against Normal and Cancer vs CIN3),
and a linear trend test over the ordered grades (Normal = 1 ... Cancer = 5). Across the 18 genes, ANOVA and trend p-values are also BH-adjusted, and the Bonferroni threshold
(0.05/18 = 0.0028) is logged. Stars above boxes show Tukey-adjusted significance versus Normal.

In [ ]:
# ==============================================================================
# Cell 05 - 18-gene panel along cervical progression (Fig. 4)
# One-way ANOVA + Tukey HSD post hoc + linear trend test over ordered grades
# ==============================================================================
PANEL <- tibble::tribble(
  ~gene, ~axis,
  "MKI67","Proliferation / DNA replication", "TOP2A","Proliferation / DNA replication",
  "MCM2","Proliferation / DNA replication",  "PCNA","Proliferation / DNA replication",
  "CDKN2A","HPV / cell cycle", "CCND1","HPV / cell cycle", "TP53","HPV / cell cycle", "RB1","HPV / cell cycle",
  "PIK3CA","Oncogenic signalling", "PTEN","Oncogenic signalling", "EGFR","Oncogenic signalling",
  "ERBB2","Oncogenic signalling", "MYC","Oncogenic signalling",
  "KRT10","Squamous differentiation", "KRT14","Squamous differentiation",
  "HIF1A","Hypoxia / angiogenesis / immune evasion", "VEGFA","Hypoxia / angiogenesis / immune evasion",
  "CD274","Hypoxia / angiogenesis / immune evasion")
stopifnot(all(PANEL$gene %in% rownames(expr)))
score <- as.numeric(group)                        # ordinal score 1..5

stats <- lapply(PANEL$gene, function(g) {
  y  <- expr[g, ]
  a  <- aov(y ~ group); p_anova <- summary(a)[[1]][["Pr(>F)"]][1]
  tk <- TukeyHSD(a)$group
  tr <- summary(lm(y ~ score))$coefficients["score", ]
  med <- tapply(y, group, median)
  tibble(gene = g, p_anova = p_anova,
         tukey_CIN1_vs_Normal = tk["CIN1-Normal", "p adj"], tukey_CIN2_vs_Normal = tk["CIN2-Normal", "p adj"],
         tukey_CIN3_vs_Normal = tk["CIN3-Normal", "p adj"], tukey_Cancer_vs_Normal = tk["Cancer-Normal", "p adj"],
         tukey_Cancer_vs_CIN3 = tk["Cancer-CIN3", "p adj"],
         trend_slope_per_grade = tr["Estimate"], trend_p = tr["Pr(>|t|)"],
         spearman_rho = suppressWarnings(cor(y, score, method = "spearman")),
         med_Normal = med["Normal"], med_CIN1 = med["CIN1"], med_CIN2 = med["CIN2"], med_CIN3 = med["CIN3"], med_Cancer = med["Cancer"])
}) %>% bind_rows() %>% mutate(fdr_anova = p.adjust(p_anova, "BH"), fdr_trend = p.adjust(trend_p, "BH")) %>%
  left_join(PANEL, by = "gene") %>% relocate(axis, .after = gene)
write.csv(stats, "results/tables/TableS4_panel_ANOVA_Tukey_trend.csv", row.names = FALSE)
for (i in seq_len(nrow(stats))) {
  log_num(paste0("panel_", stats$gene[i], "_anova_p"), signif(stats$p_anova[i], 3))
  log_num(paste0("panel_", stats$gene[i], "_trend_slope"), round(stats$trend_slope_per_grade[i], 3))
}
log_num("panel_bonferroni_threshold", round(0.05 / nrow(PANEL), 4))
print(as.data.frame(stats %>% mutate(across(where(is.numeric), ~ signif(., 3))) %>%
                      dplyr::select(gene, p_anova, fdr_anova, starts_with("tukey"), trend_slope_per_grade, trend_p)))

# ---- Plot data, per-panel annotation (ANOVA P + stars vs Normal by Tukey HSD) ----
long <- as.data.frame(t(expr[PANEL$gene, ])) %>% mutate(group = group) %>%
  pivot_longer(-group, names_to = "gene", values_to = "expr") %>%
  mutate(gene = factor(gene, levels = PANEL$gene))
ann <- stats %>% dplyr::select(gene, p_anova, starts_with("tukey_") ) %>%
  pivot_longer(c(tukey_CIN1_vs_Normal, tukey_CIN2_vs_Normal, tukey_CIN3_vs_Normal, tukey_Cancer_vs_Normal),
               names_to = "cmp", values_to = "p") %>%
  mutate(group = factor(sub("tukey_(.*)_vs_Normal", "\\1", cmp), levels = GROUP_LEVELS), lab = stars(p)) %>%
  left_join(long %>% group_by(gene) %>% summarise(ymax = max(expr), yr = diff(range(expr))), by = "gene") %>%
  mutate(y = ymax + 0.04 * yr, gene = factor(gene, levels = PANEL$gene))
hdr <- stats %>% mutate(gene = factor(gene, levels = PANEL$gene),
                        lab = sprintf("ANOVA P = %s; Cancer vs CIN3: %s", fmt_p(p_anova), stars(tukey_Cancer_vs_CIN3))) %>%
  left_join(long %>% group_by(gene) %>% summarise(ymax = max(expr), yr = diff(range(expr))), by = "gene") %>%
  mutate(y = ymax + 0.2 * yr)
p4 <- ggplot(long, aes(group, expr, fill = group)) +
  geom_boxplot(outlier.shape = NA, width = 0.65, alpha = 0.85) +
  geom_jitter(width = 0.15, size = 0.5, alpha = 0.5) +
  geom_text(data = ann, aes(x = group, y = y, label = lab), inherit.aes = FALSE, size = 3) +
  geom_text(data = hdr, aes(x = 3, y = y, label = lab), inherit.aes = FALSE, size = 2.5) +
  facet_wrap(~ gene, scales = "free_y", ncol = 6) +
  scale_fill_manual(values = GROUP_COLS, guide = "none") +
  scale_y_continuous(expand = expansion(mult = c(0.05, 0.3))) +
  labs(x = NULL, y = expression(log[2]~"expression")) + theme_paper +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))
save_fig(p4, "Fig4_panel_GSE63514", 13, 7.5)
print(p4)

## Cell 6 - TCGA-CESC: download and preparation
TCGA-CESC RNA-seq (GDC, STAR - Counts workflow, TPM) is the independent bulk-tumour dataset (TCGA Research Network, *Nature* 2017, PMID 28112728).
Selected samples: primary tumours (one aliquot per patient; n = 304) and solid-tissue normals (n = 3); the 2 metastatic samples are not used.
Duplicate gene symbols keep the Ensembl entry with the highest mean TPM; MT-/ribosomal genes are removed; **TPM is renormalised to 10^6 per sample** after removal
and log2(TPM + 1) is used downstream.

*Limitation:* only three normal samples exist in TCGA-CESC, so tumour-vs-normal comparisons have low power for effect sizes and rest on very few normals.

In [ ]:
# ==============================================================================
# Cell 06 - TCGA-CESC: GDC download (STAR - Counts workflow), sample selection,
# gene collapsing, MT-/ribosomal removal and TPM renormalisation
# ==============================================================================
suppressPackageStartupMessages({library(TCGAbiolinks); library(SummarizedExperiment)})
q <- GDCquery(project = "TCGA-CESC", data.category = "Transcriptome Profiling",
              data.type = "Gene Expression Quantification", workflow.type = "STAR - Counts")
GDCdownload(q, method = "api", files.per.chunk = 20, directory = "data/GDCdata")
se <- GDCprepare(q, directory = "data/GDCdata")
saveRDS(se, "data/tcga_cesc_se.rds")
cd <- as.data.frame(colData(se))
print(table(cd$sample_type))
log_num("tcga_n_samples_raw", ncol(se))
for (s in names(table(cd$sample_type))) log_num(paste0("tcga_n_", gsub(" ", "_", s)), sum(cd$sample_type == s))

# ---- Keep primary tumours (one aliquot per patient) and solid-tissue normals ----
keep <- cd$sample_type %in% c("Primary Tumor", "Solid Tissue Normal")
cd2 <- cd[keep, ]; cd2$barcode <- rownames(cd2)
cd2 <- cd2[order(cd2$sample_type, cd2$barcode), ]
cd2 <- cd2[!duplicated(paste(cd2$patient, cd2$sample_type)), ]
log_num("tcga_n_tumor_used", sum(cd2$sample_type == "Primary Tumor"))
log_num("tcga_n_normal_used", sum(cd2$sample_type == "Solid Tissue Normal"))
se2 <- se[, cd2$barcode]
tpm <- assay(se2, "tpm_unstrand")
rd  <- as.data.frame(rowData(se2))
gname <- rd$gene_name
ok <- !is.na(gname) & gname != ""
tpm <- tpm[ok, ]; gname <- gname[ok]
# one row per gene symbol: the Ensembl entry with the highest mean TPM
ord <- order(rowMeans(tpm), decreasing = TRUE)
tpm <- tpm[ord, ]; gname <- gname[ord]
dup <- duplicated(gname)
tpm <- tpm[!dup, ]; rownames(tpm) <- gname[!dup]
log_num("tcga_n_genes_collapsed", nrow(tpm))

# ---- Remove MT- and ribosomal-protein transcripts, then renormalise TPM -------
excl <- grepl(EXCLUDE_REGEX, rownames(tpm))
log_num("tcga_n_genes_excluded_MT_ribosomal", sum(excl))
log_num("tcga_n_MT_genes_present", sum(grepl("^MT-", rownames(tpm))))
log_num("tcga_median_pct_TPM_removed", round(median(colSums(tpm[excl, ]) / colSums(tpm) * 100), 2))
write.csv(data.frame(gene = rownames(tpm)[excl]), "results/tables/TableS_excluded_genes_TCGA.csv", row.names = FALSE)
tpm <- tpm[!excl, ]
tpm <- sweep(tpm, 2, colSums(tpm), "/") * 1e6       # TPM renormalised to 1e6 per sample
expr_t <- log2(tpm + 1)
log_num("tcga_n_genes_analysed", nrow(expr_t))
tcga_group <- factor(ifelse(cd2$sample_type == "Primary Tumor", "Tumor", "Normal"), levels = c("Normal", "Tumor"))

## Cell 7 - TCGA-CESC panel statistics and the preliminary heatmap (superseded by Cell 9, **Fig. 5**)
Tumour vs normal for the 18 genes with the Wilcoxon rank-sum test. Because 3 vs 304 is extremely unbalanced, the **exact** test is used (the normal approximation
cannot go below p ~ 0.003 here, which would make Bonferroni correction impossible by construction); the asymptotic p-value is kept in the table for comparison.
P-values are adjusted by BH and Bonferroni. The heatmap shows row-wise Z-scores truncated at +/-2, Euclidean distance and complete linkage; the dendrogram is cut in two clusters
and the sample composition of each cluster is logged.

In [ ]:
# ==============================================================================
# Cell 07 - TCGA-CESC: 18-gene panel, tumour vs normal (Wilcoxon, BH + Bonferroni)
# and hierarchical-clustering heatmap (Fig. 5)
# ==============================================================================
stopifnot(all(PANEL$gene %in% rownames(expr_t)))
tw <- lapply(PANEL$gene, function(g) {
  y <- expr_t[g, ]
  tibble(gene = g, median_normal = median(y[tcga_group == "Normal"]), median_tumor = median(y[tcga_group == "Tumor"]),
         delta_log2TPM = median(y[tcga_group == "Tumor"]) - median(y[tcga_group == "Normal"]),
         wilcox_p = wilcox.test(y ~ tcga_group, exact = TRUE)$p.value,          # exact permutation p (n = 3 vs 304)
         wilcox_p_asymptotic = wilcox.test(y ~ tcga_group, exact = FALSE)$p.value)  # normal approximation, for comparison
}) %>% bind_rows() %>% mutate(fdr = p.adjust(wilcox_p, "BH"), bonferroni = p.adjust(wilcox_p, "bonferroni")) %>%
  left_join(PANEL, by = "gene") %>% relocate(axis, .after = gene)
write.csv(tw, "results/tables/TableS5_TCGA_panel_tumor_vs_normal.csv", row.names = FALSE)
log_num("tcga_panel_n_bonferroni_sig", sum(tw$bonferroni < 0.05)); log_num("tcga_panel_n_bonferroni_sig_asymptotic", sum(p.adjust(tw$wilcox_p_asymptotic, "bonferroni") < 0.05)); log_num("tcga_panel_n_fdr_sig", sum(tw$fdr < 0.05))
n_t <- sum(tcga_group == "Tumor"); n_n <- sum(tcga_group == "Normal")
log_num("tcga_min_possible_exact_wilcoxon_p", signif(2 / choose(n_t + n_n, n_n), 3))
log_num("tcga_min_possible_bonferroni_p", signif(min(1, 18 * 2 / choose(n_t + n_n, n_n)), 3))
print(as.data.frame(tw %>% mutate(across(where(is.numeric), ~ signif(., 3))) %>% dplyr::select(gene, median_normal, median_tumor, wilcox_p, fdr, bonferroni)))

# Heatmap: row-wise Z-score truncated at +/-2, Euclidean distance, complete linkage
m <- expr_t[PANEL$gene, ]
z <- t(scale(t(m))); z[z > 2] <- 2; z[z < -2] <- -2
hc_cols <- hclust(dist(t(z), method = "euclidean"), method = "complete")   # same distance/linkage as pheatmap
cl <- cutree(hc_cols, k = 2)                                               # two sample clusters, named A and B
cl_lab <- factor(paste0("Cluster ", LETTERS[cl]))
log_num("tcga_heatmap_cluster_sizes", paste(table(cl_lab), collapse = "/"))
log_num("tcga_heatmap_normal_per_cluster", paste(table(cl_lab, tcga_group)[, "Normal"], collapse = "/"))
ann_col <- data.frame(Sample = tcga_group, Cluster = cl_lab, row.names = colnames(m))
ann_row <- data.frame(Axis = PANEL$axis, row.names = PANEL$gene)
ann_cols <- list(Sample = c(Normal = "#E69F00", Tumor = "#009E73"), Cluster = c("Cluster A" = "#CC79A7", "Cluster B" = "#56B4E9"))
write.csv(data.frame(sample = names(cl), cluster = cl, group = as.character(tcga_group)), "results/tables/Fig5_cluster_assignment.csv", row.names = FALSE)
for (ext in c("png", "tiff")) {
  f <- file.path("results/figures", paste0("Fig5_heatmap_TCGA_CESC.", ext))
  if (ext == "png") png(f, width = 3000, height = 1800, res = 300) else tiff(f, width = 3000, height = 1800, res = 300, compression = "lzw", type = "cairo")
  pheatmap(z, color = colorRampPalette(c("#2166AC", "white", "#B2182B"))(100), clustering_distance_rows = "euclidean",
           clustering_distance_cols = "euclidean", clustering_method = "complete", annotation_col = ann_col, annotation_row = ann_row, annotation_colors = ann_cols,
           show_colnames = FALSE, fontsize_row = 9, treeheight_col = 30, treeheight_row = 30)
  dev.off()
}

## Cell 8 - TCGA-CESC differential expression (limma-trend) and **Fig. 4** (volcano)
Tumour vs normal on log2(TPM + 1) with `limma` using `eBayes(trend = TRUE, robust = TRUE)` (the *limma-trend* approach for log-TPM data; Ritchie et al., *Nucleic Acids Res* 2015, PMID 25605792).
Transcripts must have TPM > 1 in at least 3 samples. The design is highly unbalanced (304 tumours vs 3 normals), so variance moderation matters and the normal arm is descriptive only.

**Interpretation caveat.** The most strongly reduced transcripts in tumours are smooth-muscle/stromal genes (*DES*, *CNN1*, *ACTG2*, *MYH11*), which suggests that the 3 normal samples differ in tissue composition from the epithelium-rich tumours. The cell also logs the agreement with the GSE63514 contrast (Spearman correlation of log2FC and sign concordance of transcripts significant in both), which is weak; the two contrasts are therefore not replicates of each other.


In [ ]:
# ==============================================================================
# Cell 08 - TCGA-CESC: differential expression, Tumour vs Normal, and volcano plot
# (manuscript Fig. 2; file Fig6_volcano_TCGA_CESC)
# Model: limma-trend on log2(TPM + 1) (Ritchie et al. 2015, Nucleic Acids Res; Law et al. 2014, Genome Biol),
# with robust empirical-Bayes moderation. Only 3 normal samples exist, so the variance moderation matters;
# results are descriptive for the normal arm. Unbalanced design: 304 tumours vs 3 normals.
# Needs in the session: expr_t, tcga_group (from cell 06) and the helpers of cell 01.
# ==============================================================================
keep <- rowSums(expr_t > 1) >= 3                      # TPM > 1 in at least 3 samples
log_num("tcga_deg_n_tested", sum(keep))
m8 <- expr_t[keep, ]
design8 <- model.matrix(~ 0 + tcga_group); colnames(design8) <- c("Normal", "Tumor")
fit8 <- lmFit(m8, design8)
fit8 <- eBayes(contrasts.fit(fit8, makeContrasts(Tumor_vs_Normal = Tumor - Normal, levels = design8)),
               trend = TRUE, robust = TRUE)
deg_t <- topTable(fit8, coef = "Tumor_vs_Normal", number = Inf, adjust.method = "BH", sort.by = "none") %>%
  tibble::rownames_to_column("gene") %>%
  dplyr::mutate(class = dplyr::case_when(adj.P.Val < FDR_CUT & logFC >  LFC_CUT ~ "Up in tumour",
                                         adj.P.Val < FDR_CUT & logFC < -LFC_CUT ~ "Down in tumour",
                                         TRUE ~ "Not significant"))
write.csv(deg_t, "results/tables/TableS6_TCGA_limma_Tumor_vs_Normal.csv", row.names = FALSE)
n_up_t <- sum(deg_t$class == "Up in tumour"); n_dn_t <- sum(deg_t$class == "Down in tumour")
log_num("tcga_deg_n_up", n_up_t); log_num("tcga_deg_n_down", n_dn_t)
log_num("tcga_deg_logFC_min", round(min(deg_t$logFC), 2)); log_num("tcga_deg_logFC_max", round(max(deg_t$logFC), 2))
log_num("tcga_deg_max_neglog10FDR", round(max(-log10(deg_t$adj.P.Val)), 2))
cat(sprintf("TCGA tumour vs normal: %d up, %d down (FDR<%.2f, |log2FC|>%d), %d genes tested\n",
            n_up_t, n_dn_t, FDR_CUT, LFC_CUT, nrow(deg_t)))
# Concordance with the cervical progression data (GSE63514, Cancer vs Normal), genes in common
if (exists("deg")) {
  cm <- merge(deg[, c("gene", "logFC", "class")], deg_t[, c("gene", "logFC", "class")], by = "gene", suffixes = c("_gse", "_tcga"))
  log_num("tcga_vs_gse_n_common_genes", nrow(cm))
  log_num("tcga_vs_gse_logFC_spearman", round(cor(cm$logFC_gse, cm$logFC_tcga, method = "spearman"), 3))
  both <- cm$class_gse != "Not significant" & cm$class_tcga != "Not significant"
  log_num("tcga_vs_gse_n_both_sig", sum(both))
  log_num("tcga_vs_gse_n_both_sig_same_direction", sum(both & sign(cm$logFC_gse) == sign(cm$logFC_tcga)))
}
lab_t <- bind_rows(deg_t %>% filter(class == "Up in tumour")   %>% slice_max(logFC, n = 10),
                   deg_t %>% filter(class == "Down in tumour") %>% slice_min(logFC, n = 10))
write.csv(lab_t, "results/tables/Fig6_labelled_genes.csv", row.names = FALSE)
cols_t <- c("Up in tumour" = "#D62728", "Down in tumour" = "#1F77B4", "Not significant" = "grey70")
p6 <- ggplot(deg_t, aes(logFC, -log10(adj.P.Val), colour = class)) +
  geom_point(size = 0.5, alpha = 0.5) +
  geom_vline(xintercept = c(-LFC_CUT, LFC_CUT), linetype = "dashed", colour = "grey40") +
  geom_hline(yintercept = -log10(FDR_CUT), linetype = "dashed", colour = "grey40") +
  geom_text_repel(data = lab_t, aes(label = gene), size = 2.8, colour = "black", fontface = "italic",
                  max.overlaps = Inf, box.padding = 0.35, segment.size = 0.25, min.segment.length = 0) +
  scale_colour_manual(values = cols_t, name = NULL) +
  labs(x = expression(log[2]~"fold change (Tumour / Normal)"), y = expression(-log[10]~"FDR")) +
  theme_paper + theme(legend.position = "bottom")
save_fig(p6, "Fig6_volcano_TCGA_CESC", 7, 5.5)
print(p6)

## Cell 9 - TCGA-CESC clinical descriptors, annotated heatmap (**Fig. 5**) and panel by clinical group
Replaces the age-distribution figure criticised by the reviewer with cohort descriptors (Table 3: histology, FIGO stage, HPV genotype, age, vital status; TCGA 2017 HPV call, `paper_CLIN.HPV_Hcall`).
The 18-gene heatmap (row z-scores truncated at +/-2, Euclidean distance, complete linkage, two clusters) is annotated with histology, FIGO stage and HPV genotype.
Associations between clusters and clinical variables use Fisher's exact test; panel transcripts by histology, FIGO stage and HPV genotype use Kruskal-Wallis with BH adjustment over the 18 genes.
Missing annotations are shown in grey and excluded from each test. Needs `expr_t`, `cd2`, `tcga_group` from Cell 6.

In [ ]:
# ==============================================================================
# Cell 09 - TCGA-CESC clinical descriptors, annotated panel heatmap (Fig. 5) and
# panel expression by histology / FIGO stage / HPV genotype.
# Needs from the session: expr_t (log2 TPM, MT-/ribosomal removed, renormalised),
# cd2 (colData aligned to the columns of expr_t), tcga_group, PANEL, log_num, save_fig, fmt_p.
# ==============================================================================
suppressPackageStartupMessages({library(rstatix)})
stopifnot(identical(rownames(cd2), colnames(expr_t)) || all(cd2$barcode == colnames(expr_t)))
is_tum <- tcga_group == "Tumor"
tum <- cd2[is_tum, ]
fig_grp <- function(x) { x <- sub("^Stage ", "", x)
  ifelse(is.na(x), NA, ifelse(grepl("^IV", x), "IV", ifelse(grepl("^III", x), "III", ifelse(grepl("^II", x), "II", ifelse(grepl("^I", x), "I", NA))))) }
tum$FIGO <- factor(fig_grp(tum$figo_stage), levels = c("I", "II", "III", "IV"))
tum$Histology <- factor(dplyr::case_when(grepl("^Adenosquamous", tum$primary_diagnosis) ~ "Adenosquamous",
                                         grepl("denocarcinoma", tum$primary_diagnosis) ~ "Adenocarcinoma",
                                         grepl("quamous", tum$primary_diagnosis) ~ "Squamous", TRUE ~ NA_character_),
                        levels = c("Squamous", "Adenocarcinoma", "Adenosquamous"))
hc_call <- tum[["paper_CLIN.HPV_Hcall"]]
tum$HPV <- factor(dplyr::case_when(is.na(hc_call) ~ NA_character_, hc_call == "HPV16" ~ "HPV16", hc_call == "HPV18" ~ "HPV18",
                                   hc_call == "negative" ~ "HPV-negative", TRUE ~ "Other HPV types"),
                  levels = c("HPV16", "HPV18", "Other HPV types", "HPV-negative"))
tum$age <- as.numeric(tum$age_at_index)

# ---- Cohort descriptors (Table 3) ---------------------------------------------
tb <- function(v, nm) { t <- table(v, useNA = "ifany"); data.frame(variable = nm, level = ifelse(is.na(names(t)), "missing", names(t)), n = as.integer(t), value = NA_character_) }
desc <- rbind(
  data.frame(variable = "Primary tumours", level = "n", n = nrow(tum), value = NA_character_),
  data.frame(variable = "Age (years)", level = "mean (range)", n = NA_integer_, value = sprintf("%.1f (%d-%d)", mean(tum$age, na.rm = TRUE), min(tum$age, na.rm = TRUE), max(tum$age, na.rm = TRUE))),
  tb(tum$Histology, "Histology"), tb(tum$FIGO, "FIGO stage"), tb(tum$HPV, "HPV genotype"), tb(tum$vital_status, "Vital status"))
write.csv(desc, "results/tables/Table3_TCGA_cohort_descriptors.csv", row.names = FALSE)
print(desc)
log_num("tcga_age_mean", round(mean(tum$age, na.rm = TRUE), 1)); log_num("tcga_age_min", min(tum$age, na.rm = TRUE)); log_num("tcga_age_max", max(tum$age, na.rm = TRUE))
for (l in levels(tum$Histology)) log_num(paste0("tcga_hist_n_", l), sum(tum$Histology == l, na.rm = TRUE))
for (l in levels(tum$FIGO)) log_num(paste0("tcga_FIGO_n_", l), sum(tum$FIGO == l, na.rm = TRUE)); log_num("tcga_FIGO_missing", sum(is.na(tum$FIGO)))
for (l in levels(tum$HPV)) log_num(paste0("tcga_HPV_n_", l), sum(tum$HPV == l, na.rm = TRUE)); log_num("tcga_HPV_annotated", sum(!is.na(tum$HPV)))

# ---- Annotated heatmap: same 18-gene panel / distance / linkage / k = 2 as Fig. 5 ---
m <- expr_t[PANEL$gene, ]
z <- t(scale(t(m))); z[z > 2] <- 2; z[z < -2] <- -2
hc_cols <- hclust(dist(t(z), method = "euclidean"), method = "complete")
cl <- cutree(hc_cols, k = 2)
cl_lab <- factor(paste0("Cluster ", LETTERS[cl]))
ann_col <- data.frame(Sample = tcga_group, Cluster = cl_lab, row.names = colnames(m))
ann_col$Histology <- NA; ann_col$FIGO <- NA; ann_col$HPV <- NA
ann_col[rownames(tum), "Histology"] <- as.character(tum$Histology)
ann_col[rownames(tum), "FIGO"] <- as.character(tum$FIGO)
ann_col[rownames(tum), "HPV"] <- as.character(tum$HPV)
ann_col$Histology <- factor(ann_col$Histology, levels = levels(tum$Histology)); ann_col$FIGO <- factor(ann_col$FIGO, levels = levels(tum$FIGO)); ann_col$HPV <- factor(ann_col$HPV, levels = levels(tum$HPV))
ann_row <- data.frame(Axis = PANEL$axis, row.names = PANEL$gene)
ann_cols <- list(Sample = c(Normal = "#E69F00", Tumor = "#009E73"), Cluster = c("Cluster A" = "#CC79A7", "Cluster B" = "#56B4E9"),
                 Histology = c(Squamous = "#33a02c", Adenocarcinoma = "#6a3d9a", Adenosquamous = "#ff7f00"),
                 FIGO = c(I = "#deebf7", II = "#9ecae1", III = "#fc9272", IV = "#a50f15"),
                 HPV = c(HPV16 = "#d73027", HPV18 = "#fc8d59", `Other HPV types` = "#fee090", `HPV-negative` = "#91bfdb"))
for (ext in c("png", "tiff")) {
  f <- file.path("results/figures", paste0("Fig7_heatmap_TCGA_clinical.", ext))
  if (ext == "png") png(f, width = 3200, height = 2100, res = 300) else tiff(f, width = 3200, height = 2100, res = 300, compression = "lzw", type = "cairo")
  pheatmap::pheatmap(z, color = colorRampPalette(c("#2166AC", "white", "#B2182B"))(100), clustering_distance_rows = "euclidean",
           clustering_distance_cols = "euclidean", clustering_method = "complete", annotation_col = ann_col, annotation_row = ann_row,
           annotation_colors = ann_cols, show_colnames = FALSE, fontsize_row = 9, treeheight_col = 30, treeheight_row = 30)
  dev.off()
}
# Association of the two clusters with each annotation (tumours only; Fisher exact, simulated p for tables larger than 2 x 2)
cl_t <- cl_lab[is_tum]
assoc <- function(a) { ok <- !is.na(a); tb <- table(cl_t[ok], droplevels(a[ok]))
  list(p = fisher.test(tb, simulate.p.value = TRUE, B = 1e5)$p.value, n = sum(ok), tb = tb) }
a_h <- assoc(tum$Histology); a_f <- assoc(tum$FIGO); a_v <- assoc(tum$HPV)
log_num("tcga_cluster_vs_histology_p", signif(a_h$p, 2)); log_num("tcga_cluster_vs_FIGO_p", signif(a_f$p, 2)); log_num("tcga_cluster_vs_HPV_p", signif(a_v$p, 2))
log_num("tcga_cluster_vs_histology_n", a_h$n); log_num("tcga_cluster_vs_FIGO_n", a_f$n); log_num("tcga_cluster_vs_HPV_n", a_v$n)
tabs <- list(histology = a_h$tb, FIGO = a_f$tb, HPV = a_v$tb)
cl_tab <- do.call(rbind, lapply(names(tabs), function(n) { d <- as.data.frame(tabs[[n]], stringsAsFactors = FALSE); names(d) <- c("cluster", "level", "n"); cbind(annotation = n, d) }))
write.csv(cl_tab, "results/tables/TableS7b_cluster_by_clinical_counts.csv", row.names = FALSE)
write.csv(data.frame(annotation = c("histology", "FIGO", "HPV"), n = c(a_h$n, a_f$n, a_v$n), fisher_p = c(a_h$p, a_f$p, a_v$p)), "results/tables/TableS7_cluster_vs_clinical_tests.csv", row.names = FALSE)
cat("\nCluster vs annotation:\n"); print(tabs); cat("p:", a_h$p, a_f$p, a_v$p, "\n")

# ---- Panel genes by histology, FIGO and HPV genotype (Kruskal-Wallis, BH across 18 genes; Dunn BH post hoc) ---
et <- expr_t[PANEL$gene, is_tum]
dc <- data.frame(t(et), check.names = FALSE); dc$Histology <- tum$Histology; dc$FIGO <- tum$FIGO; dc$HPV <- tum$HPV
long_c <- tidyr::pivot_longer(dc, all_of(PANEL$gene), names_to = "gene", values_to = "log2tpm")
kw_all <- do.call(rbind, lapply(c("Histology", "FIGO", "HPV"), function(v) {
  d <- long_c[!is.na(long_c[[v]]), ]; d$g <- droplevels(d[[v]])
  r <- d |> dplyr::group_by(gene) |> dplyr::summarise(n = dplyr::n(), KW_p = kruskal.test(log2tpm ~ g)$p.value, .groups = "drop")
  r$KW_q_BH <- p.adjust(r$KW_p, "BH"); r$factor <- v; r }))
write.csv(kw_all, "results/tables/TableS8_panel_by_clinical_KW.csv", row.names = FALSE)
for (v in c("Histology", "FIGO", "HPV")) { s <- kw_all[kw_all$factor == v, ]; log_num(paste0("tcga_panel_KW_q_lt_0.05_", v), sum(s$KW_q_BH < 0.05)); log_num(paste0("tcga_panel_KW_genes_q_lt_0.05_", v), paste(s$gene[s$KW_q_BH < 0.05], collapse = ",")) }
print(as.data.frame(kw_all |> dplyr::filter(KW_q_BH < 0.05) |> dplyr::arrange(factor, KW_q_BH)), digits = 3)


## Cell 10 - TCGA-CESC overall survival (**Fig. 6**)
Univariable Cox models (Cox 1972) on z-scored log2(TPM+1) for the 18 panel transcripts plus PDCD1, CTLA4, FOXP3 and CD8A (hazard ratio per SD; BH over 22 models), Kaplan-Meier curves with a median split and log-rank test,
and a CD274 Cox model adjusted for age, stage (I-II vs III-IV) and histology, with the proportional-hazards check (`cox.zph`). Event = death; living patients are censored at last follow-up; follow-up is short and the models are exploratory.

In [ ]:
# ==============================================================================
# Cell 10 - TCGA-CESC overall survival: CD274 (pre-specified) and the 18-gene panel + checkpoint/TME genes.
# Event = death; time = days to death, otherwise days to last follow-up (months = days / 30.4375).
# Cox models use z-scored log2 TPM (HR per +1 SD); BH within each model family; Kaplan-Meier on the median split.
# Needs c09 objects: tum (tumour clinical table with FIGO/Histology/age), expr_t, tcga_group, PANEL.
# ==============================================================================
suppressPackageStartupMessages({library(survival); library(survminer); library(patchwork)})
is_tum <- tcga_group == "Tumor"
sv <- data.frame(time_d = ifelse(tum$vital_status == "Dead", as.numeric(tum$days_to_death), as.numeric(tum$days_to_last_follow_up)),
                 event = as.integer(tum$vital_status == "Dead"), age = tum$age,
                 stage = factor(ifelse(tum$FIGO %in% c("I", "II"), "I-II", ifelse(tum$FIGO %in% c("III", "IV"), "III-IV", NA)), levels = c("I-II", "III-IV")),
                 hist = factor(ifelse(tum$Histology == "Squamous", "SCC", ifelse(is.na(tum$Histology), NA, "Non-SCC")), levels = c("SCC", "Non-SCC")),
                 row.names = rownames(tum))
sv$time_m <- sv$time_d / 30.4375
ok <- !is.na(sv$time_d) & sv$time_d > 0
log_num("surv_n_tumours", nrow(sv)); log_num("surv_n_used", sum(ok)); log_num("surv_events", sum(sv$event[ok])); log_num("surv_n_excluded", sum(!ok))
log_num("surv_median_followup_months", round(median(sv$time_m[ok & sv$event == 0]), 1))

genes_s <- unique(c(PANEL$gene, "CD274", "PDCD1", "CTLA4", "FOXP3", "CD8A"))
stopifnot(all(genes_s %in% rownames(expr_t)))
Xs <- t(scale(t(expr_t[genes_s, is_tum])))               # z-score across tumours
dsv <- cbind(sv[, c("time_m", "event", "age", "stage", "hist")], t(Xs))[ok, ]
uni <- do.call(rbind, lapply(genes_s, function(g) {
  s <- summary(coxph(as.formula(paste0("Surv(time_m, event) ~ `", g, "`")), data = dsv))
  data.frame(gene = g, HR = s$conf.int[1, "exp(coef)"], lo = s$conf.int[1, "lower .95"], hi = s$conf.int[1, "upper .95"], p = s$coefficients[1, "Pr(>|z|)"]) }))
uni$q_BH <- p.adjust(uni$p, "BH"); uni <- uni[order(uni$p), ]
stopifnot(all(uni$lo <= uni$HR & uni$HR <= uni$hi))
write.csv(uni, "results/tables/TableS9_Cox_univariable_HR_per_SD.csv", row.names = FALSE)
log_num("cox_genes_tested", nrow(uni)); log_num("cox_genes_q_lt_0.05", sum(uni$q_BH < 0.05)); log_num("cox_genes_p_lt_0.05", sum(uni$p < 0.05))
for (i in 1:5) log_num(paste0("cox_top", i), paste0(uni$gene[i], " HR ", round(uni$HR[i], 2), " (", round(uni$lo[i], 2), "-", round(uni$hi[i], 2), "); p = ", signif(uni$p[i], 2), "; q = ", signif(uni$q_BH[i], 2)))
log_num("cox_CD274_uni", paste0("HR ", round(uni$HR[uni$gene == "CD274"], 2), " (", round(uni$lo[uni$gene == "CD274"], 2), "-", round(uni$hi[uni$gene == "CD274"], 2), "); p = ", signif(uni$p[uni$gene == "CD274"], 2)))
print(uni, digits = 3)

# Adjusted model for CD274: age, FIGO group (I-II vs III-IV) and histology (SCC vs non-SCC); complete cases
dadj <- dsv[complete.cases(dsv[, c("age", "stage", "hist", "CD274")]), ]
fit_adj <- coxph(Surv(time_m, event) ~ CD274 + age + stage + hist, data = dadj)
sa <- summary(fit_adj)
adj <- data.frame(term = rownames(sa$coefficients), HR = sa$conf.int[, "exp(coef)"], lo = sa$conf.int[, "lower .95"], hi = sa$conf.int[, "upper .95"], p = sa$coefficients[, "Pr(>|z|)"])
write.csv(adj, "results/tables/TableS10_Cox_CD274_adjusted.csv", row.names = FALSE)
log_num("cox_adj_n", nrow(dadj)); log_num("cox_adj_events", sum(dadj$event))
log_num("cox_adj_CD274", paste0("HR ", round(adj$HR[1], 2), " (", round(adj$lo[1], 2), "-", round(adj$hi[1], 2), "); p = ", signif(adj$p[1], 2)))
log_num("cox_adj_stage", paste0("HR ", round(adj$HR[3], 2), " (", round(adj$lo[3], 2), "-", round(adj$hi[3], 2), "); p = ", signif(adj$p[3], 2)))
ph <- cox.zph(fit_adj); log_num("cox_adj_PH_global_p", signif(ph$table["GLOBAL", "p"], 2))
print(adj, digits = 3); print(ph)

# Kaplan-Meier, median split, log-rank (all genes, BH)
km_p <- sapply(genes_s, function(g) { grp <- ifelse(dsv[[g]] >= median(dsv[[g]]), "High", "Low"); 1 - pchisq(survdiff(Surv(dsv$time_m, dsv$event) ~ grp)$chisq, 1) })
km_tab <- data.frame(gene = genes_s, logrank_p = km_p, q_BH = p.adjust(km_p, "BH")); write.csv(km_tab, "results/tables/TableS11_KM_logrank.csv", row.names = FALSE)
log_num("km_CD274_logrank_p", signif(km_p["CD274"], 2)); log_num("km_genes_q_lt_0.05", sum(km_tab$q_BH < 0.05))

# ---- Figure 6: (A) KM CD274, (B) forest of univariable HR per SD -----------------
grp_cd <- factor(ifelse(dsv$CD274 >= median(dsv$CD274), "High", "Low"), levels = c("High", "Low"))
dk <- data.frame(dsv, grp_cd); fit_cd <- survfit(Surv(time_m, event) ~ grp_cd, data = dk)
pA <- ggsurvplot(fit_cd, data = dk, palette = c("#d73027", "#4575b4"), conf.int = FALSE, censor.size = 2, xlab = "Time (months)", ylab = "Overall survival",
                 legend.title = "", legend.labs = c("CD274 high", "CD274 low"), legend = c(0.75, 0.9), ggtheme = theme_classic(base_size = 9), size = 0.6)$plot +
  annotate("text", x = 0, y = 0.08, hjust = 0, size = 2.6, label = paste0("log-rank p = ", fmt_p(km_p["CD274"]), "\nn = ", nrow(dsv), ", deaths = ", sum(dsv$event)))
fp <- uni; fp$gene <- factor(fp$gene, levels = rev(fp$gene)); fp$sig <- ifelse(fp$q_BH < 0.05, "q < 0.05", "q >= 0.05")
pB <- ggplot(fp, aes(HR, gene, colour = sig)) + geom_vline(xintercept = 1, linetype = "dashed", colour = "grey40", linewidth = 0.3) +
  geom_pointrange(aes(xmin = lo, xmax = hi), size = 0.25, linewidth = 0.4) + scale_x_log10() +
  scale_colour_manual(values = c("q < 0.05" = "#d73027", "q >= 0.05" = "#555555"), name = NULL) +
  labs(x = "Hazard ratio per +1 SD (95% CI)", y = NULL) + theme_classic(base_size = 9) + theme(axis.text.y = element_text(face = "italic", size = 6), legend.position = "bottom")
fig8 <- pA + pB + plot_layout(widths = c(1, 1.1)) + plot_annotation(tag_levels = "A") & theme(plot.tag = element_text(face = "bold", size = 10))
save_fig(fig8, "Fig8_survival_TCGA_CESC", 7.2, 4.6)


## Cell 11 - TCGA-CESC DNA methylation, Illumina HM450 (**Fig. 7**)
GDC SeSAMe level-3 beta values (312 files: 307 primary tumours, 3 solid normal, 2 metastases not used). Filters: <=5% missing, no sex chromosomes, no SNP-flagged probes (GPL13534 annotation), leaving 316,546 probes.
M-values (offset 0.001; Du et al. 2010) are median-centred and tested with limma (`eBayes(robust = TRUE)`, tumour minus normal). DMP: adjusted P < 0.05 and |log2FC| > 1. Reports delta-beta, CpG-island context and promoter probes of the panel genes
(TSS200, TSS1500, 5'UTR, first exon). **Slow step**: the download takes >1 h and the beta-value matrix (~1 GB) is cached in `data/`. Only three normal samples are available, so the contrast is exploratory.

In [ ]:
# ==============================================================================
# Cell 11 - TCGA-CESC DNA methylation (Illumina HM450, GDC SeSAMe level-3 beta values):
# tumour (01) vs solid-tissue normal (11) differentially methylated probes (DMPs), limma on M-values.
# Download is cached in data/TCGA_CESC_HM450_beta.rds (about 4 GB of raw files). Probe annotation: GEO GPL13534.
# ==============================================================================
suppressPackageStartupMessages({library(TCGAbiolinks); library(GEOquery); library(patchwork)})
rds_meth <- "data/TCGA_CESC_HM450_beta.rds"
if (!file.exists(rds_meth)) {
  filt <- list(op = "and", content = list(
    list(op = "=", content = list(field = "cases.project.project_id", value = "TCGA-CESC")),
    list(op = "=", content = list(field = "data_type", value = "Methylation Beta Value")),
    list(op = "=", content = list(field = "platform", value = "Illumina Human Methylation 450"))))
  body <- jsonlite::toJSON(list(filters = filt, size = 1000, format = "JSON", fields = "file_id,file_name,cases.samples.submitter_id,cases.samples.sample_type"), auto_unbox = TRUE)
  hits <- httr::content(httr::POST("https://api.gdc.cancer.gov/files", httr::content_type_json(), body = body), as = "parsed")$data$hits
  rs <- data.frame(file_id = vapply(hits, function(h) h$file_id, ""), file_name = vapply(hits, function(h) h$file_name, ""),
                   barcode = vapply(hits, function(h) h$cases[[1]]$samples[[1]]$submitter_id, ""), stringsAsFactors = FALSE)
  cat("GDC HM450 files:", nrow(rs), "\n"); stopifnot(!anyDuplicated(rs$barcode))
  dest <- "data/GDCdata/TCGA-CESC/DNA_Methylation/Methylation_Beta_Value"
  fl <- file.path(dest, rs$file_id, rs$file_name)
  if (!all(file.exists(fl))) {
    q_meth <- GDCquery(project = "TCGA-CESC", data.category = "DNA Methylation", data.type = "Methylation Beta Value", platform = "Illumina Human Methylation 450")
    GDCdownload(q_meth, method = "api", files.per.chunk = 10, directory = "data/GDCdata")
  }
  stopifnot(all(file.exists(fl)))
  p0 <- data.table::fread(fl[1], header = FALSE, col.names = c("probe", "beta"))$probe
  beta_all <- matrix(NA_real_, nrow = length(p0), ncol = length(fl), dimnames = list(p0, rs$barcode))
  for (i in seq_along(fl)) { d <- data.table::fread(fl[i], header = FALSE, col.names = c("probe", "beta")); beta_all[, i] <- d$beta[match(p0, d$probe)] }
  saveRDS(beta_all, rds_meth); rm(d); gc()
}
beta_all <- readRDS(rds_meth)
stype <- substr(colnames(beta_all), 14, 15)          # 01 primary tumour, 11 solid-tissue normal, 06 metastasis (not used)
log_num("meth_samples_raw", ncol(beta_all)); log_num("meth_tumour", sum(stype == "01")); log_num("meth_normal", sum(stype == "11")); log_num("meth_excluded_other_types", sum(!stype %in% c("01", "11")))
beta_all <- beta_all[, stype %in% c("01", "11")]; stype <- stype[stype %in% c("01", "11")]
if (!file.exists("data/GPL13534.soft.gz")) GEOquery::getGEO("GPL13534", destdir = "data", AnnotGPL = FALSE)
gpl <- Table(getGEO(filename = "data/GPL13534.soft.gz", AnnotGPL = FALSE)); gpl <- gpl[match(rownames(beta_all), gpl$ID), ]
s1 <- rowMeans(is.na(beta_all)) <= 0.05; b1 <- beta_all[s1, ]; g1 <- gpl[s1, ]
s2 <- !(g1$CHR %in% c("X", "Y")); b2 <- b1[s2, ]; g2 <- g1[s2, ]
snp <- (!is.na(g2$Probe_SNPs) & g2$Probe_SNPs != "") | (!is.na(g2$Probe_SNPs_10) & g2$Probe_SNPs_10 != "")
b3 <- b2[!snp, ]; g3 <- g2[!snp, ]
cnt <- c(raw = nrow(beta_all), na_le_5pct = nrow(b1), no_sex_chr = nrow(b2), no_SNP = nrow(b3)); print(cnt)
for (n in names(cnt)) log_num(paste0("meth_probes_", n), cnt[[n]])
beta2m <- function(b, off = 0.001) { b <- pmax(pmin(b, 1 - off), off); log2(b / (1 - b)) }
M <- beta2m(b3); med <- apply(M, 2, median, na.rm = TRUE); M <- sweep(M, 2, med - median(med), "-")     # median-centred M-values
grp <- factor(ifelse(stype == "01", "Tumour", "Normal"), levels = c("Normal", "Tumour"))
des <- model.matrix(~ 0 + grp); colnames(des) <- levels(grp)
fit <- eBayes(contrasts.fit(lmFit(M, des), makeContrasts(Tumour - Normal, levels = des)), robust = TRUE)
dmp <- topTable(fit, number = Inf, adjust.method = "BH", sort.by = "P")
dmp$status <- "Not significant"; dmp$status[dmp$adj.P.Val < 0.05 & dmp$logFC > 1] <- "Hypermethylated"; dmp$status[dmp$adj.P.Val < 0.05 & dmp$logFC < -1] <- "Hypomethylated"
dmp$probe <- rownames(dmp); gi <- match(dmp$probe, g3$ID); dmp$gene <- g3$UCSC_RefGene_Name[gi]; dmp$island_relation <- g3$Relation_to_UCSC_CpG_Island[gi]
dmp$delta_beta <- rowMeans(b3[dmp$probe, grp == "Tumour"], na.rm = TRUE) - rowMeans(b3[dmp$probe, grp == "Normal"], na.rm = TRUE)
write.csv(dmp, "results/tables/TableS12_TCGA_CESC_DMPs_all_probes.csv", row.names = FALSE)
log_num("dmp_n_tumour", sum(grp == "Tumour")); log_num("dmp_n_normal", sum(grp == "Normal"))
log_num("dmp_tested", nrow(dmp)); log_num("dmp_hyper", sum(dmp$status == "Hypermethylated")); log_num("dmp_hypo", sum(dmp$status == "Hypomethylated"))
log_num("dmp_hyper_dbeta_gt_0.2", sum(dmp$status == "Hypermethylated" & dmp$delta_beta > 0.2)); log_num("dmp_hypo_dbeta_lt_-0.2", sum(dmp$status == "Hypomethylated" & dmp$delta_beta < -0.2))
print(table(dmp$status))
# Methylation of the panel genes (promoter-region probes: TSS200/TSS1500/5'UTR/1stExon)
reg <- g3$UCSC_RefGene_Group[match(dmp$probe, g3$ID)]
prom <- grepl("TSS200|TSS1500|5'UTR|1stExon", reg)
pg <- sapply(strsplit(dmp$gene, ";"), function(x) any(x %in% PANEL$gene))
pan <- dmp[pg & prom & dmp$status != "Not significant", c("probe", "gene", "logFC", "delta_beta", "adj.P.Val", "status")]
write.csv(pan, "results/tables/TableS13_panel_gene_promoter_DMPs.csv", row.names = FALSE)
log_num("dmp_panel_promoter_sig", nrow(pan)); log_num("dmp_panel_promoter_genes", paste(sort(unique(unlist(strsplit(pan$gene, ";")))[unique(unlist(strsplit(pan$gene, ";"))) %in% PANEL$gene]), collapse = ","))
print(pan, digits = 3)

# ---- Figure 9: (A) volcano, (B) CpG-island context ------------------------------------
cols_m <- c(Hypermethylated = "#b2182b", Hypomethylated = "#2166ac", `Not significant` = "#b0b0b0"); dmp$status <- factor(dmp$status, levels = names(cols_m))
pA <- ggplot(dmp, aes(logFC, -log10(adj.P.Val), colour = status)) + geom_point(size = 0.25, alpha = 0.5, stroke = 0) +
  geom_vline(xintercept = c(-1, 1), linetype = "dashed", colour = "grey40", linewidth = 0.3) + geom_hline(yintercept = -log10(0.05), linetype = "dashed", colour = "grey40", linewidth = 0.3) +
  scale_colour_manual(values = cols_m, name = NULL, labels = function(x) paste0(sub("^Hyper.*", "Hyper", sub("^Hypo.*", "Hypo", sub("^Not.*", "NS", x))), " (n = ", format(table(dmp$status)[x], big.mark = ",", trim = TRUE), ")")) +
  labs(x = "M-value difference (tumour - normal)", y = expression(-log[10]~"adjusted p")) + guides(colour = guide_legend(override.aes = list(size = 2, alpha = 1), nrow = 1)) +
  theme_classic(base_size = 9) + theme(legend.position = "bottom")
ctx <- function(x) factor(ifelse(x == "" | is.na(x), "Open sea", sub("^N_|^S_", "", x)), levels = c("Island", "Shore", "Shelf", "Open sea"))
isl <- dmp |> dplyr::mutate(ctx = ctx(island_relation)); a1 <- isl |> dplyr::filter(status != "Not significant") |> dplyr::count(status, ctx); a0 <- isl |> dplyr::count(ctx) |> dplyr::mutate(status = "All tested")
isl_all <- dplyr::bind_rows(a1 |> dplyr::mutate(status = as.character(status)), a0) |> dplyr::group_by(status) |> dplyr::mutate(pct = 100 * n / sum(n)) |> dplyr::ungroup() |>
  dplyr::mutate(status = factor(status, levels = c("All tested", "Hypermethylated", "Hypomethylated")))
write.csv(isl_all, "results/tables/TableS14_DMP_CpG_island_context.csv", row.names = FALSE)
for (s in c("All tested", "Hypermethylated", "Hypomethylated")) log_num(paste0("dmp_island_pct_", gsub(" ", "_", s)), round(isl_all$pct[isl_all$status == s & isl_all$ctx == "Island"], 1))
pB <- ggplot(isl_all, aes(status, pct, fill = ctx)) + geom_col(width = 0.7, colour = "black", linewidth = 0.2) +
  scale_fill_manual(values = c(Island = "#1b7837", Shore = "#7fbf7b", Shelf = "#d9f0d3", `Open sea` = "#f7f7f7"), name = NULL) +
  labs(x = NULL, y = "Probes (%)") + theme_classic(base_size = 9) + theme(axis.text.x = element_text(angle = 25, hjust = 1), legend.position = "bottom")
fig9 <- pA + pB + plot_layout(widths = c(1.4, 1)) + plot_annotation(tag_levels = "A") & theme(plot.tag = element_text(face = "bold", size = 10))
save_fig(fig9, "Fig9_methylation_TCGA_CESC", 7.2, 3.6)


## Cell 12 - Sensitivity of the panel to the probe-collapsing rule (**Table S15**)
Main analysis keeps the probe with the highest mean per gene. This cell repeats the panel ANOVA/Tukey after collapsing by the highest variance and reports whether the Bonferroni classification of each gene changes.
It rebuilds `sym`, `keep` and `group` because earlier cells may have reused those names. It also logs the sample-characteristic fields available in GSE63514 (no smoking or alcohol information).

In [ ]:
# ==============================================================================
# Cell 12 - Sensitivity of the 18-transcript panel (GSE63514) to the probe-collapsing rule
# Main analysis: highest mean probe per gene. Alternative: highest-variance probe per gene (rule used in the earlier draft for another series).
# Needs from the session: eset, expr_probe, keep, sym, group, GROUP_LEVELS, PANEL, EXCLUDE_REGEX, stats (from cell 05), log_num.
# ==============================================================================
cat("Sample-characteristic fields available in the GSE63514 series matrix:\n"); print(grep("ch1$", colnames(pData(eset)), value = TRUE))
log_num("gse63514_characteristic_fields", paste(sub(":ch1$", "", grep("ch1$", colnames(pData(eset)), value = TRUE)), collapse = "; "))
# Objects below are rebuilt here because later cells (c10/c11) may have overwritten names such as keep/group in the live session
sym  <- trimws(fData(eset)[["Gene symbol"]]); keep <- !is.na(sym) & sym != "" & !grepl("///", sym, fixed = TRUE)
tt <- pData(eset)[["tissue type:ch1"]]
group <- factor(dplyr::case_when(grepl("normal", tt) ~ "Normal", grepl("low grade", tt) ~ "CIN1", grepl("moderate grade", tt) ~ "CIN2",
                                 grepl("high grade", tt) ~ "CIN3", grepl("cancer", tt) ~ "Cancer"), levels = GROUP_LEVELS)
stopifnot(length(keep) == nrow(expr_probe), length(group) == ncol(expr_probe), all(c("gene","p_anova","tukey_Cancer_vs_Normal") %in% names(stats)))
pt <- tibble(probe = rownames(expr_probe)[keep], symbol = sym[keep], v = matrixStats::rowVars(expr_probe[keep, ]))
bv <- pt %>% group_by(symbol) %>% slice_max(v, n = 1, with_ties = FALSE) %>% ungroup()
ev <- expr_probe[bv$probe, ]; rownames(ev) <- bv$symbol; ev <- ev[!grepl(EXCLUDE_REGEX, rownames(ev)), ]
bm <- tibble(probe = rownames(expr_probe)[keep], symbol = sym[keep], m = rowMeans(expr_probe[keep, ])) %>% group_by(symbol) %>% slice_max(m, n = 1, with_ties = FALSE) %>% ungroup()
same_probe <- bv$probe %in% bm$probe[match(bv$symbol, bm$symbol)]
log_num("probe_rule_genes_same_probe_pct", round(100 * mean(bv$probe == bm$probe[match(bv$symbol, bm$symbol)]), 1))
sens <- lapply(PANEL$gene, function(g) {
  y <- ev[g, ]; a <- aov(y ~ group); tk <- TukeyHSD(a)$group
  tibble(gene = g, probe_highest_mean = bm$probe[bm$symbol == g], probe_highest_variance = bv$probe[bv$symbol == g],
         anova_p_mean = stats$p_anova[stats$gene == g], anova_p_variance = summary(a)[[1]][["Pr(>F)"]][1],
         tukey_Cancer_vs_Normal_mean = stats$tukey_Cancer_vs_Normal[stats$gene == g], tukey_Cancer_vs_Normal_variance = tk["Cancer-Normal", "p adj"]) }) %>% bind_rows()
sens$bonf_sig_mean <- sens$anova_p_mean < 0.05 / 18; sens$bonf_sig_variance <- sens$anova_p_variance < 0.05 / 18
write.csv(sens, "results/tables/TableS15_panel_probe_rule_sensitivity.csv", row.names = FALSE)
log_num("probe_rule_panel_same_probe", sum(sens$probe_highest_mean == sens$probe_highest_variance))
log_num("probe_rule_bonferroni_sig_mean", sum(sens$bonf_sig_mean)); log_num("probe_rule_bonferroni_sig_variance", sum(sens$bonf_sig_variance))
log_num("probe_rule_bonferroni_changed", paste(sens$gene[sens$bonf_sig_mean != sens$bonf_sig_variance], collapse = ","))
print(as.data.frame(sens %>% mutate(across(where(is.numeric), ~ signif(., 3)))))


## Cell 13 - Registry of numbers quoted in the manuscript, and session information
Every number written in the Results section comes from `NUM_LOG`; this is the file to check when verifying the text against the analysis.

In [ ]:
nl <- data.frame(key = names(NUM_LOG), value = vapply(NUM_LOG, function(x) paste(x, collapse = ";"), ""))
write.csv(nl, "results/tables/numbers_logged_all.csv", row.names = FALSE)
print(nl, row.names = FALSE)
sessionInfo()
